# EDA with SQL
The Falcon 9 launch table is loaded into an SQLite database and queried from the notebook.

In [1]:
import re, os, sqlite3
import pandas as pd

In [2]:
SRC = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv'
raw = pd.read_csv('Spacex.csv') if os.path.exists('Spacex.csv') else pd.read_csv(SRC)
raw.columns = [re.sub(r'[^0-9A-Za-z_]+', '_', c.strip()).strip('_') for c in raw.columns]
raw['Date'] = pd.to_datetime(raw['Date'].astype(str).str.strip(), format='mixed').dt.strftime('%Y-%m-%d')
print(list(raw.columns))
raw.head()

['Date', 'Time_UTC', 'Booster_Version', 'Launch_Site', 'Payload', 'PAYLOAD_MASS__KG', 'Orbit', 'Customer', 'Mission_Outcome', 'Landing_Outcome']


,Date,Time_UTC,Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [3]:
print(list(raw.columns))
names = {'mass': 'PAYLOAD_MASS__KG_', 'landing': 'Landing_Outcome', 'mission': 'Mission_Outcome',
         'booster': 'Booster_Version', 'customer': 'Customer'}
for col in list(raw.columns):
    low = col.lower()
    if 'launch' in low and 'site' in low:
        raw = raw.rename(columns={col: 'Launch_Site'})
    for key, new in names.items():
        if key in low:
            raw = raw.rename(columns={col: new})
print(list(raw.columns))
con = sqlite3.connect('my_data1.db')
raw.to_sql('SPACEXTBL', con, if_exists='replace', index=False)
def q(sql):
    print(sql.strip()); print()
    return pd.read_sql(sql, con)

['Date', 'Time_UTC', 'Booster_Version', 'Launch_Site', 'Payload', 'PAYLOAD_MASS__KG', 'Orbit', 'Customer', 'Mission_Outcome', 'Landing_Outcome']
['Date', 'Time_UTC', 'Booster_Version', 'Launch_Site', 'Payload', 'PAYLOAD_MASS__KG_', 'Orbit', 'Customer', 'Mission_Outcome', 'Landing_Outcome']


## Task 1 - Unique launch site names

In [4]:
q("SELECT DISTINCT Launch_Site FROM SPACEXTBL")

SELECT DISTINCT Launch_Site FROM SPACEXTBL



,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


## Task 2 - 5 records where launch site begins with 'CCA'

In [5]:
q("SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5")

SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5



,Date,Time_UTC,Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


## Task 3 - Total payload mass carried by NASA (CRS) boosters

In [6]:
q("SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_NASA_CRS FROM SPACEXTBL WHERE Customer = 'NASA (CRS)'")

SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_NASA_CRS FROM SPACEXTBL WHERE Customer = 'NASA (CRS)'



,Total_Payload_NASA_CRS
0,45596


## Task 4 - Average payload mass for booster version F9 v1.1

In [7]:
q("SELECT AVG(PAYLOAD_MASS__KG_) AS Avg_Payload_F9_v1_1 FROM SPACEXTBL WHERE Booster_Version = 'F9 v1.1'")

SELECT AVG(PAYLOAD_MASS__KG_) AS Avg_Payload_F9_v1_1 FROM SPACEXTBL WHERE Booster_Version = 'F9 v1.1'



,Avg_Payload_F9_v1_1
0,2928.4


## Task 5 - Date of first successful landing on a ground pad

In [8]:
q("SELECT MIN(Date) AS First_Ground_Pad_Success FROM SPACEXTBL WHERE Landing_Outcome = 'Success (ground pad)'")

SELECT MIN(Date) AS First_Ground_Pad_Success FROM SPACEXTBL WHERE Landing_Outcome = 'Success (ground pad)'



,First_Ground_Pad_Success
0,2015-12-22


## Task 6 - Boosters with drone-ship success and payload between 4000 and 6000 kg

In [9]:
q("SELECT DISTINCT Booster_Version FROM SPACEXTBL WHERE Landing_Outcome = 'Success (drone ship)' AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000")

SELECT DISTINCT Booster_Version FROM SPACEXTBL WHERE Landing_Outcome = 'Success (drone ship)' AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000



,Booster_Version
0,F9 FT B1022
1,F9 FT B1026
2,F9 FT B1021.2
3,F9 FT B1031.2


## Task 7 - Total successful and failed mission outcomes

In [10]:
q("SELECT Mission_Outcome, COUNT(*) AS Total FROM SPACEXTBL GROUP BY Mission_Outcome")

SELECT Mission_Outcome, COUNT(*) AS Total FROM SPACEXTBL GROUP BY Mission_Outcome



,Mission_Outcome,Total
0,Failure (in flight),1
1,Success,98
2,Success,1
3,Success (payload status unclear),1


## Task 8 - Boosters that carried the maximum payload mass

In [11]:
q("SELECT DISTINCT Booster_Version FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)")

SELECT DISTINCT Booster_Version FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)



,Booster_Version
0,F9 B5 B1048.4
1,F9 B5 B1049.4
2,F9 B5 B1051.3
3,F9 B5 B1056.4
4,F9 B5 B1048.5
5,F9 B5 B1051.4
6,F9 B5 B1049.5
7,F9 B5 B1060.2
8,F9 B5 B1058.3
9,F9 B5 B1051.6


## Task 9 - 2015 failed drone-ship landings (month, outcome, booster, site)

In [12]:
q("SELECT substr(Date, 6, 2) AS Month, Landing_Outcome, Booster_Version, Launch_Site FROM SPACEXTBL WHERE Landing_Outcome = 'Failure (drone ship)' AND substr(Date, 1, 4) = '2015'")

SELECT substr(Date, 6, 2) AS Month, Landing_Outcome, Booster_Version, Launch_Site FROM SPACEXTBL WHERE Landing_Outcome = 'Failure (drone ship)' AND substr(Date, 1, 4) = '2015'



,Month,Landing_Outcome,Booster_Version,Launch_Site
0,01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


## Task 10 - Rank landing outcomes between 2010-06-04 and 2017-03-20

In [13]:
q("SELECT Landing_Outcome, COUNT(*) AS Total FROM SPACEXTBL WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY Total DESC")

SELECT Landing_Outcome, COUNT(*) AS Total FROM SPACEXTBL WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY Total DESC



,Landing_Outcome,Total
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1
